In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()  # 加载.env文件里的变量
print(os.getenv("DEEPSEEK_API_KEY"))  # 现在可以正常读取了
print(os.getenv("LANGSMITH_TRACING"))
print(os.getenv("LANGSMITH_ENDPOINT"))
print(os.getenv("LANGSMITH_API_KEY"))
print(os.getenv("LANGSMITH_PROJECT"))

In [ ]:
from langgraph.graph.message import add_messages
import getpass
import os
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langgraph.graph import StateGraph,START,END
from typing_extensions import TypedDict
from typing import Annotated

class State(TypedDict):
    messages: Annotated[list,add_messages]

graph_buider=StateGraph(State)

llm = ChatOpenAI(
        model="deepseek-chat",  # 使用的模型名称，目前官方推荐用 'deepseek-chat'
        api_key=os.getenv("DEEPSEEK_API_KEY"),  # 你的 DeepSeek API Key
        base_url="https://api.deepseek.com/v1",  # DeepSeek API 地址
        temperature=0,
    )

def chatbot(state:State):
    return {"messages":[llm.invoke(state["messages"])]}

graph_buider.add_node("chatbot",chatbot)
graph_buider.add_edge(START,"chatbot")
graph_buider.add_edge("chatbot",END)

graph=graph_buider.compile()

In [ ]:
def stream_graph_updates(user_input:str):
    for event in  graph.stream({"messages":[("user",user_input)]}):
        for value in event.values():
            print("模型回复:",value["messages"][-1].content)

while True:
    try:
        user_input=input("用户提问：")
        if user_input.lower() in ["退出"]:
            print("再见")
            break
        stream_graph_updates(user_input)
    except:
        user_input="what do you know about langGraph?"
        print("User: "+user_input)
        stream_graph_updates(user_input)
        break